# End-to-end Ginsu application

_____________________________
This demo notebook is split in 2 parts:

- **Machine Learning modelling**

This part implements a basic regressor on the [California housing dataset](https://www.openml.org/search?type=data&sort=runs&id=41211&status=active) to predict house values.
  
- **Model debugging with Ginsu**

This part identifies slices where the training error of the model is significantly higher, thanks to [sliceline](https://github.com/DataDome/sliceline).

## Machine Learning modelling

We used a [HistGradientBoostingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html) with default parameters as regressor. The optimisation metric is the [Root Mean Square Error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html).

No preprocessing or feature engineering is applied in the pipeline. It is not the purpose of this demo notebook.

The training error is the element-wise square error.

In [ ]:
# import useful modules
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import HistGradientBoostingRegressor

pd.set_option("display.max_rows", None, "display.max_columns", None)

# fetch California housing dataset
X, y = fetch_california_housing(as_frame=True, return_X_y=True)

# define the model
model = HistGradientBoostingRegressor(random_state=42)

# training
model.fit(X, y)

# predict
y_pred = model.predict(X)

# compute element-wise square error (the lower, the better)
training_errors = (y - y_pred) ** 2

## Model debbuging with Ginsu

**Ginsu considers all the columns of the input dataset as categorical.**

So, to get more relevant slices, features should be discretized.

Indeed, columns as-is would lead to poor exploitable results. We would rather have range of values to specific value in our slices definition.

To discretize them and compute their bins, we use [OptBinning](http://gnpalencia.org/optbinning/) but feel free to experiment other binning implementations.

Ginsu configuration:
- `alpha = 0.95`: we are interested in small slice with high log loss.
- `k = 1`: we want Ginsu to find the rules with the best score.
- `max_l = 4`: we limit the maximum lattice level to balance between slice expressiveness and performance.
- `min_sup = 1`: because the input dataset is relatively small, we do not add constraint regarding the minimal support.

In [ ]:
# import Ginsu and binning class
from ginsu import Slicefinder
from optbinning import ContinuousOptimalBinning

# Columns have to be bined
optimal_binner = ContinuousOptimalBinning(max_n_bins=5)

X_trans = pd.DataFrame(
    np.array(
        [
            optimal_binner.fit_transform(
                X[col], training_errors, metric="bins"
            )
            for col in X.columns
        ]
    ).T,
    columns=X.columns,
)

# fitting Ginsu
sf = Slicefinder(alpha=0.95, k=1, max_l=4, min_sup=1, verbose=True)

sf.fit(X_trans, training_errors)

In [ ]:
# slices found
pd.DataFrame(
    sf.top_slices_,
    columns=sf.feature_names_in_,
    index=sf.get_feature_names_out(),
)

**Note:**

We found 1 slices with `k` set to 1.

_(`None` values refer to unused features in each slices.)_

In [ ]:
from sklearn.metrics import mean_squared_error as mse

# select one slice
slice_index = 0
current_slice = sf.top_slices_[slice_index]

# create a pandas filter
predicate_conditions = [
    X_trans[feature_name] == feature_value
    for feature_name, feature_value in zip(sf.feature_names_in_, current_slice)
    if feature_value is not None
]
condition = " & ".join(
    [f"@predicate_conditions[{i}]" for i in range(len(predicate_conditions))]
)

# get slice element indices
indices = X_trans.query(condition).index

print("Model MSE on:")
print(f"- the full dataset ({X.shape[0]} houses):", mse(y, y_pred))
print(
    f"- the selected slice ({len(indices)} houses):",
    mse(y.iloc[indices], y_pred[indices]),
)

# Conclusion

With Ginsu, we identified a subset of 1756 houses on which the model performs significantly worse. Those houses:
- count 2 or less average number of household members (`AveOccup='(-inf, 2.02)'`).

To improve the modelisation, we should focus on reducing the error on those houses.